### How to Tune variables

| Variable | What it is | How to change it |
|---|---|---|
| `SIGMA_NORM` | How many standard deviations from the mean to set limits for **Normal** and **Log-Normal** distributions. A higher value = wider limits (fewer outliers flagged). A lower value = tighter limits (more outliers flagged). | Change the number on this line: `SIGMA_NORM = 3` |
| `SIGMA_C` | How many standard deviations from the mean to set limits for **Logistic**, **Weibull**, and **No Clear Fit** distributions. Same logic as above. | Change the number on this line: `SIGMA_C = 3` |


---
## Import neccessary libary and defining global variable 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
from distfit import distfit
from scipy.stats import shapiro

SIGMA_NORM = 3
SIGMA_C = 3

---
## Load Data
Reads your dataset from `dataset.csv` and automatically separates columns into two groups:
- **Continuous columns** (e.g. height, weight, blood pressure) — these will be analysed for outliers.
- **Categorical columns** (e.g. gender, plaque) — these will be skipped for analysis.

### Note:
- Make sure your file is named `dataset.csv` and is in the **same folder** as this notebook.
- Alternatively, change `"dataset.csv"` to your filename on this line: `df = pd.read_csv("dataset.csv")`

In [ ]:
df = pd.read_csv("dataset.csv") # change file name here

categorical_cols = []
numeric_cols = []

for col in df.columns:

    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

numeric_df = df[numeric_cols]

---
## Visualise Data Distribution

For every numeric column, this creates:
1. **Histogram** — shows the shape of your data.
2. **Boxplot** — shows outliers as red dots beyond the whiskers.
    * where whisker is calculated by:
        1. Lower whisker = smallest value >= Q1 - 1.5×IQR
        2. Upper whisker = largest value <= Q3 + 1.5×IQR

*Null values are ignored because new data entries should not have missing values.

In [ ]:
for col in numeric_df.columns:

    data = numeric_df[col].dropna()

    if len(data) < 3:
        print(f"Skipping {col}: Insufficient data")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(22, 6))

    # Histogram
    sns.histplot(
        data,
        bins=30,
        kde=True,
        edgecolor='black',
        color='skyblue',
        ax=axes[0]
    )

    axes[0].set_title(f'Histogram / Distribution of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frequency')
    axes[0].grid(False)

    # Boxplot
    sns.boxplot(
        x=data,
        color='skyblue',
        flierprops={
            "markerfacecolor": "red",
            "marker": "o"
        },
        ax=axes[1]
    )

    axes[1].set_title(f'Boxplot of {col}')
    axes[1].set_xlabel(col)
    axes[1].grid(False)

    plt.tight_layout()
    plt.show()

---
## Find the Best-Fit Distribution for Each Column

For each numeric column, this code:
1. Applies **hard floors** — biological minimums to filter out typos (e.g., Hip < 60cm).
2. Tests 4 candidate distributions: **Normal**, **Logistic**, **Log-Normal**, and **Weibull**.
3. Ranks them using `distfit` (a library that finds the best statistical distribution).
4. Verifies the top candidate with a **Kolmogorov-Smirnov (KS) test** (if the p-value > 0.05, the distribution is a good fit).
5. Picks the **best fit** for each column (lowest KS statistic with p-value > 0.05).

### How to tune

| Variable | What it controls | Where to change it |
|---|---|---|
| `hard_floors` | Minimum possible values for specific columns. Values below this are treated as typos and removed. | In the `hard_floors = {...}` dictionary. Add or edit entries like `"Waist": 40`. |
| `candidate_distr` | Which distributions to test. You can add or remove distribution names. | In the list: `['norm', 'logistic', 'lognorm', 'weibull_min']` |


In [ ]:

# Define hard biological minimums to clear out typos 
# https://stacks.cdc.gov/view/cdc/174595
hard_floors = {
    "Waist" : 40,
    "Hip": 60
}
results = []
candidate_distr = [
    'norm',
    'logistic',
    'lognorm',
    'weibull_min'
]

for col in numeric_df.columns:

    data = numeric_df[col].dropna().values

    if col in hard_floors:
        data = data[data >= hard_floors[col]]

    # skip columns with insufficient data or no variation because distfit will fail to fit distributions
    if len(data) < 10:
        print(f"Skipping {col} (insufficient data)")
        continue

    if np.std(data) == 0:
        print(f"Skipping {col} (no data variation)")
        continue

    dfit = distfit(
        distr=candidate_distr,
        verbose=0
    )

    dfit.fit_transform(data)

    # get top 3 distributions
    sorted_summary = dfit.summary.sort_values("score")

    # print(f"\nAnalyzing column: {col}")
    # print("\nDistributions rank:")
    # print(sorted_summary[['name', 'score']])

    # do ks-test for verification of the best fit
    for _, row in sorted_summary.iterrows():

        dist_name = row['name']

        try:
            dist = getattr(stats, dist_name)
            params = dist.fit(data)

            # Compare actual data against the theoretical distribution curve using the KS test
            ks_stat, ks_pvalue = stats.kstest(
                rvs=data,
                cdf=lambda x: dist.cdf(x, *params)
            )

            results.append({
                'Column': col,
                'Distribution': dist_name,
                'KS Statistic': ks_stat,
                'KS P-Value': ks_pvalue,
                'Parameters': params
            })

        except Exception as e:
            print(f"KS test failed for {dist_name}: {e}")

    # print("\nBest Distribution based on distfit:")
    # print(dfit.model)
    # print(dfit.stats) # confirm what stats use for distfit matching

results_df = pd.DataFrame(results)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

results_df['KS Statistic'] = results_df['KS Statistic'].round(4)
results_df['KS P-Value'] = results_df['KS P-Value'].round(4)

# print("\nSummary of Best Fits for All Columns (ks-test):")
# print(results_df)

# get best fit for each column based on the lowest KS statistic
bestfit_results = []
for col, group in results_df.groupby('Column'):
    
    # filter out distributions with p-value <= 0.05 (reject null hypothesis)
    valid_group = group[group['KS P-Value'] > 0.05]
    
    if valid_group.empty:
        bestfit_results.append({
            'Column': col,
            'Distribution': 'No clear fit',
            'KS Statistic': np.nan,
            'KS P-Value': np.nan,
            'Parameters': None
        })

    else:
        best_row = valid_group.loc[
            valid_group['KS Statistic'].idxmin()
        ]
        bestfit_results.append({
            'Column': col,
            'Distribution': best_row['Distribution'],
            'KS Statistic': best_row['KS Statistic'],
            'KS P-Value': best_row['KS P-Value'],
            'Parameters': best_row['Parameters']
        })


bestfit_df = pd.DataFrame(bestfit_results)
print("\nSummary of Best Fits for All Columns (after KS-test verification):")
print(bestfit_df)

---
## Visualise the Fitted Distributions

For each column that had a clear distribution fit, this creates:
1. **Histogram + Fitted Curve** — shows your actual data as bars and the theoretical distribution as a red line. If the line follows the bars closely, the fit is good.
2. **Q-Q Plot** — if points follow the red line, the distribution is a good fit.

Columns with "No clear fit" are skipped.

In [ ]:
for _, row in bestfit_df.iterrows():

    col = row['Column']
    dist_name = row['Distribution']

    if dist_name == 'No clear fit':
        continue
    
    data = numeric_df[col].dropna().values

    if col in hard_floors:
        data = data[data >= hard_floors[col]]

    dist = getattr(stats, dist_name)
    params = row['Parameters']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Histogram + fitted curve
    axes[0].hist(
        data,
        bins=20,
        density=True,
        alpha=0.6,
        color='skyblue',
        edgecolor='black'
    )

    x = np.linspace(min(data), max(data), 1000)

    axes[0].plot(
        x,
        dist.pdf(x, *params),
        'r-',
        lw=2
    )

    axes[0].set_title(f'{col}\nHistogram ({dist_name})')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Density')

    # Q-Q Plot
    stats.probplot(
        data,
        dist=dist,
        sparams=params,
        plot=axes[1]
    )

    axes[1].set_title(f'{col}\nQ-Q Plot ({dist_name})')

    plt.tight_layout()
    plt.show()


---
## Calculate Lower & Upper Limits + Detect Outliers

### What this does
For each column:
1. Calculates **Lower Limit** and **Upper Limit** based on the best-fit distribution:
   - **Normal**: `mean ± SIGMA_NORM × standard deviation`
   - **Logistic**: `loc ± SIGMA_C × ((scale × π) / √3)`
   - **Log-Normal**: `exp(meanlog ± SIGMA_NORM × sdlog)`
   - **Weibull**: `mean ± SIGMA_C × std` (capped at the location parameter)
   - **No clear fit**: `mean ± SIGMA_C × std` (Chebyshev's inequality)
2. Counts how many data points fall **outside** these limits (outliers).
3. Saves everything to **`Distribution_Summary.csv`**.

### How to tune

| What to change | Effect | Where to change it |
|---|---|---|
| `SIGMA_NORM` | Wider/narrower limits for Normal & Log-Normal columns | At the top of the notebook |
| `SIGMA_C` | Wider/narrower limits for Logistic, Weibull & No Clear Fit columns | At the top of the notebook |
| `hard_floors` | Add/remove minimum values to filter typos | In the `hard_floors` dictionary |


In [ ]:
summary_results = []

for _, row in bestfit_df.iterrows():

    col = row['Column']
    dist_name = row['Distribution']
    params = row['Parameters']
    ks_stats = row['KS Statistic']
    ks_p_value = row['KS P-Value']

    data = df[col].dropna()

    if col in hard_floors:
        data = data[data >= hard_floors[col]]

    if len(data) < 3:
        continue

    param_text = None
    lower_limit = np.nan
    upper_limit = np.nan
    n_outliers = np.nan

    if dist_name != "No clear fit" and params is not None:

        # Normal
        if dist_name == "norm":

            mean, sd = params

            param_text = (
                f"mean = {mean:.4f}, "
                f"sd = {sd:.4f}"
            )

            lower_limit = mean - SIGMA_NORM * sd
            upper_limit = mean + SIGMA_NORM * sd

        # Logistic
        elif dist_name == "logistic":

            loc, scale = params

            param_text = (
                f"loc = {loc:.4f}, "
                f"scale = {scale:.4f}"
            )

            sd = scale * np.pi / np.sqrt(3)

            lower_limit = loc - SIGMA_C * sd
            upper_limit = loc + SIGMA_C * sd

        # Lognormal
        elif dist_name == "lognorm":

            log_data = np.log(data)

            meanlog = np.mean(log_data)
            sdlog = np.std(log_data)


            param_text = (
                f"meanlog = {meanlog:.4f}, "
                f"sdlog = {sdlog:.4f}"
            )

            lower_limit = np.exp(meanlog - SIGMA_NORM * sdlog)
            upper_limit = np.exp(meanlog + SIGMA_NORM * sdlog)

        # Weibull
        elif dist_name == "weibull_min":

            shape, loc, scale = params

            param_text = (
                f"shape = {shape:.4f}, "
                f"loc = {loc:.4f}, "
                f"scale = {scale:.4f}"
            )

            mean = stats.weibull_min.mean(shape, loc=loc, scale=scale)
            sd = stats.weibull_min.std(shape, loc=loc, scale=scale)

            lower_limit = mean - SIGMA_C * sd
            upper_limit = mean + SIGMA_C * sd

            # Weibull cannot be below loc; X >= loc 
            lower_limit = max(lower_limit, loc)

    # no clear fit, use mean ± kSD for outlier detection (Chebyshev's inequality)
    else:
        mean = np.mean(data)
        sd = np.std(data)
        lower_limit = mean - SIGMA_C * sd
        upper_limit = mean + SIGMA_C * sd

    # count outliers
    outliers = data[
        (data < lower_limit) |
        (data > upper_limit)
    ]
    
    n_outliers = len(outliers)

    summary_results.append({
        "Variable": col,
        "N": len(data),
        "ks_D": ks_stats,
        "ks_p_value": ks_p_value,
        "Best_fit_distribution": dist_name,
        "Parameters": param_text,
        "Lower_Limit": round(lower_limit, 4) if pd.notna(lower_limit) else np.nan,
        "Upper_Limit": round(upper_limit, 4) if pd.notna(upper_limit) else np.nan,
        "N_Outliers": n_outliers
    })

summary_df = pd.DataFrame(summary_results)

# add categorical columns to the summary with a note that they were skipped
if categorical_cols:
    skip_df = pd.DataFrame({
        'Variable': categorical_cols,
        'N': np.nan,
        'ks_D': np.nan,
        'ks_p_value': np.nan,
        'Best_fit_distribution': 'Categorical data skipped',
        'Parameters': np.nan,
        'Lower_Limit': np.nan,
        'Upper_Limit': np.nan,
        'N_Outliers': np.nan
    })

    summary_df = pd.concat(
        [summary_df, skip_df],
        ignore_index=True
    )

# print(summary_df)

try:
    summary_df.to_csv(
    "Distribution_Summary.csv",
    index=False
    )
    print("\nDistribution summary saved to 'Distribution_Summary.csv'.")
except Exception as e:
    print(f"Error saving summary to CSV: {e}")
